# IEEE Publication Research Pipeline: Parkinson's Disease Detection
### 6-Model Benchmark, Leak-Free SMOTENC Cross-Validation, & Feature Ablation
This notebook reproduces the complete experimental suite for peer-reviewed IEEE submission:
1. **6-Model Benchmark Suite** (Logistic Regression, KNN, Decision Tree, SVM, Random Forest, XGBoost)
2. **Leak-Free 5-Fold Stratified Cross-Validation** (SMOTENC applied strictly inside folds)
3. **Comprehensive Feature-Ablation Study** (Leave-one-out and single-feature evaluations)
4. **Publication-Ready Figures & LaTeX Tables** (300 DPI ROC curves, confusion matrices, SHAP beeswarm)
5. **Streamlit Artifact Synchronization**


In [ ]:
import os
import ast
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTENC
import joblib
import shap

warnings.filterwarnings('ignore')
%matplotlib inline
print('All libraries loaded successfully.')


## 1. Dataset Loading & Feature Isolation
Parses stringified symptom columns and isolates the six clinical markers.


In [ ]:
data_path = os.path.join('..', 'Dataset', 'parkinsons_disease_data_cls.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('Dataset', 'parkinsons_disease_data_cls.csv')

raw_df = pd.read_csv(data_path)
raw_df['Symptoms'] = raw_df['Symptoms'].apply(ast.literal_eval)
symptoms_df = pd.json_normalize(raw_df['Symptoms']).replace({'Yes': 1, 'No': 0})

FEATURE_COLS = ['UPDRS', 'FunctionalAssessment', 'Tremor', 'MoCA', 'Bradykinesia', 'Rigidity']
CAT_INDICES = [2, 4, 5]  # Tremor, Bradykinesia, Rigidity are binary nominal
CONT_COLS = ['UPDRS', 'FunctionalAssessment', 'MoCA']

df = pd.concat([
    raw_df[['UPDRS', 'FunctionalAssessment', 'MoCA', 'Diagnosis']],
    symptoms_df[['Tremor', 'Bradykinesia', 'Rigidity']]
], axis=1)

X = df[FEATURE_COLS].copy()
y = df['Diagnosis'].values

print(f'Total Cohort Size : {len(df)}')
print(f'Healthy (Class 0) : {sum(y==0)} ({sum(y==0)/len(y)*100:.2f}%)')
print(f"Parkinson's (1)   : {sum(y==1)} ({sum(y==1)/len(y)*100:.2f}%)")


## 2. 6-Model Benchmark Suite
Defines six diverse algorithms spanning linear, instance-based, kernel, and tree ensemble families.


In [ ]:
def get_models():
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2),
        'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42),
        'Support Vector Machine': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=200, min_samples_split=5, max_features='sqrt', random_state=42),
        'XGBoost': XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, eval_metric='logloss', random_state=42)
    }


## 3. Leak-Free 5-Fold Stratified Cross-Validation with SMOTENC
Pre-scaling and `SMOTENC` are executed strictly inside each training fold to eliminate data leakage.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = list(get_models().keys())
metrics_dict = {m: {'acc': [], 'sens': [], 'spec': [], 'prec': [], 'f1': [], 'auc': [], 'mcc': []} for m in model_names}
roc_storage = {m: {'fpr': [], 'tpr': [], 'aucs': []} for m in model_names}
cm_storage = {m: np.zeros((2, 2), dtype=int) for m in model_names}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train_f, y_train_f = X.iloc[train_idx].copy(), y[train_idx]
    X_val_f, y_val_f = X.iloc[val_idx].copy(), y[val_idx]

    fold_scaler = MinMaxScaler()
    X_train_sc = pd.DataFrame(fold_scaler.fit_transform(X_train_f), columns=FEATURE_COLS)
    X_val_sc = pd.DataFrame(fold_scaler.transform(X_val_f), columns=FEATURE_COLS)

    smote = SMOTENC(categorical_features=CAT_INDICES, random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train_f)

    models = get_models()
    for name, model in models.items():
        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_val_sc)
        y_prob = model.predict_proba(X_val_sc)[:, 1]

        cm = confusion_matrix(y_val_f, y_pred)
        cm_storage[name] += cm
        tn, fp, fn, tp = cm.ravel()

        metrics_dict[name]['acc'].append(accuracy_score(y_val_f, y_pred))
        metrics_dict[name]['sens'].append(recall_score(y_val_f, y_pred))
        metrics_dict[name]['spec'].append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        metrics_dict[name]['prec'].append(precision_score(y_val_f, y_pred))
        metrics_dict[name]['f1'].append(f1_score(y_val_f, y_pred))
        metrics_dict[name]['auc'].append(roc_auc_score(y_val_f, y_prob))
        metrics_dict[name]['mcc'].append(matthews_corrcoef(y_val_f, y_pred))

        fpr, tpr, _ = roc_curve(y_val_f, y_prob)
        roc_storage[name]['fpr'].append(fpr)
        roc_storage[name]['tpr'].append(tpr)

print('Stratified 5-Fold Cross-Validation Completed Successfully.')


## 4. Model Comparison Table (Table 1)
Displays all 6 models across 7 performance metrics (Mean ± Std).


In [ ]:
table1_rows = []
for name in model_names:
    m = metrics_dict[name]
    table1_rows.append({
        'Model': name,
        'Accuracy (%)': f"{np.mean(m['acc'])*100:.2f} ± {np.std(m['acc'])*100:.2f}",
        'Sensitivity (%)': f"{np.mean(m['sens'])*100:.2f} ± {np.std(m['sens'])*100:.2f}",
        'Specificity (%)': f"{np.mean(m['spec'])*100:.2f} ± {np.std(m['spec'])*100:.2f}",
        'Precision (%)': f"{np.mean(m['prec'])*100:.2f} ± {np.std(m['prec'])*100:.2f}",
        'F1-Score': f"{np.mean(m['f1']):.4f} ± {np.std(m['f1']):.4f}",
        'ROC-AUC': f"{np.mean(m['auc']):.4f} ± {np.std(m['auc']):.4f}",
        'MCC': f"{np.mean(m['mcc']):.4f} ± {np.std(m['mcc']):.4f}"
    })

df_table1 = pd.DataFrame(table1_rows)
display(df_table1)


## 5. Multi-Model ROC Curves & Confusion Matrices


In [ ]:
plt.figure(figsize=(8, 6), dpi=120)
colors = sns.color_palette('tab10', len(model_names))
for i, name in enumerate(model_names):
    mean_auc = np.mean(metrics_dict[name]['auc'])
    mean_fpr = np.linspace(0, 1, 100)
    tprs = [np.interp(mean_fpr, roc_storage[name]['fpr'][f], roc_storage[name]['tpr'][f]) for f in range(5)]
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
    plt.plot(mean_fpr, mean_tpr, color=colors[i], lw=2, label=f"{name} (AUC = {mean_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Chance (AUC = 0.500)')
plt.xlabel('False Positive Rate (1 - Specificity)', fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontweight='bold')
plt.title('Stratified 5-Fold Mean ROC Curves', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), dpi=120)
for idx, m_name in enumerate(['Random Forest', 'XGBoost', 'Support Vector Machine']):
    cm = cm_storage[m_name]
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    annot = np.array([[f"{cm[r,c]}\n({cm_norm[r,c]*100:.1f}%)" for c in range(2)] for r in range(2)])
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['No PD (0)', 'PD (1)'], yticklabels=['No PD (0)', 'PD (1)'])
    axes[idx].set_title(m_name, fontweight='bold')
    axes[idx].set_xlabel('Predicted Label')
    if idx == 0: axes[idx].set_ylabel('True Label')
plt.suptitle('Aggregated 5-Fold Confusion Matrices', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Feature-Ablation Study (Table 2)
Evaluates Random Forest across 13 feature subsets to quantify the value of each clinical marker.


In [ ]:
ablation_configs = {
    'Full Set (All 6 Features)': FEATURE_COLS,
    'Drop UPDRS (Prodromal Mode)': [f for f in FEATURE_COLS if f != 'UPDRS'],
    'Drop FunctionalAssessment': [f for f in FEATURE_COLS if f != 'FunctionalAssessment'],
    'Drop Tremor': [f for f in FEATURE_COLS if f != 'Tremor'],
    'Drop MoCA': [f for f in FEATURE_COLS if f != 'MoCA'],
    'Drop Bradykinesia': [f for f in FEATURE_COLS if f != 'Bradykinesia'],
    'Drop Rigidity': [f for f in FEATURE_COLS if f != 'Rigidity'],
    'Only UPDRS': ['UPDRS'],
    'Only FunctionalAssessment': ['FunctionalAssessment'],
    'Only Tremor': ['Tremor'],
    'Only MoCA': ['MoCA'],
    'Only Bradykinesia': ['Bradykinesia'],
    'Only Rigidity': ['Rigidity']
}

ablation_results = []
for config_name, feats in ablation_configs.items():
    X_sub = df[feats].copy()
    cat_idxs = [i for i, f in enumerate(feats) if f in ['Tremor', 'Bradykinesia', 'Rigidity']]
    accs, senss, specs, f1s, aucs = [], [], [], [], []

    for train_idx, val_idx in skf.split(X_sub, y):
        X_tr, y_tr = X_sub.iloc[train_idx].copy(), y[train_idx]
        X_va, y_va = X_sub.iloc[val_idx].copy(), y[val_idx]

        s = MinMaxScaler()
        X_tr_sc = pd.DataFrame(s.fit_transform(X_tr), columns=feats)
        X_va_sc = pd.DataFrame(s.transform(X_va), columns=feats)

        if len(feats) > 1 and len(cat_idxs) > 0 and len(cat_idxs) < len(feats):
            sm = SMOTENC(categorical_features=cat_idxs, random_state=42)
            X_tr_res, y_tr_res = sm.fit_resample(X_tr_sc, y_tr)
        elif len(feats) > 1 and len(cat_idxs) == 0:
            from imblearn.over_sampling import SMOTE
            sm = SMOTE(random_state=42)
            X_tr_res, y_tr_res = sm.fit_resample(X_tr_sc, y_tr)
        else:
            X_tr_res, y_tr_res = X_tr_sc, y_tr

        rf = RandomForestClassifier(n_estimators=200, min_samples_split=5, random_state=42)
        rf.fit(X_tr_res, y_tr_res)
        yp = rf.predict(X_va_sc)
        yprob = rf.predict_proba(X_va_sc)[:, 1]

        accs.append(accuracy_score(y_va, yp))
        senss.append(recall_score(y_va, yp))
        f1s.append(f1_score(y_va, yp))
        aucs.append(roc_auc_score(y_va, yprob))

    ablation_results.append({
        'Configuration': config_name,
        'Features': len(feats),
        'Accuracy (%)': f"{np.mean(accs)*100:.2f} ± {np.std(accs)*100:.2f}",
        'Sensitivity (%)': f"{np.mean(senss)*100:.2f} ± {np.std(senss)*100:.2f}",
        'F1-Score': f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
        'ROC-AUC': f"{np.mean(aucs):.4f} ± {np.std(aucs):.4f}"
    })

df_ablation = pd.DataFrame(ablation_results)
display(df_ablation)


## 7. Global TreeSHAP Feature Attribution (Beeswarm)


In [ ]:
clean_scaler = MinMaxScaler()
X_scaled = pd.DataFrame(clean_scaler.fit_transform(X), columns=FEATURE_COLS)
smote_full = SMOTENC(categorical_features=CAT_INDICES, random_state=42)
X_res, y_res = smote_full.fit_resample(X_scaled, y)

rf_full = RandomForestClassifier(n_estimators=200, min_samples_split=5, random_state=42)
rf_full.fit(X_res, y_res)

explainer = shap.TreeExplainer(rf_full)
shap_vals = explainer.shap_values(X_scaled)
if isinstance(shap_vals, list):
    shap_vals_pos = shap_vals[1]
elif shap_vals.ndim == 3:
    shap_vals_pos = shap_vals[:, :, 1]
else:
    shap_vals_pos = shap_vals

plt.figure(figsize=(8.5, 5), dpi=120)
shap.summary_plot(shap_vals_pos, X, feature_names=FEATURE_COLS, show=False)
plt.title('Global Clinical Risk Factor Attribution (TreeSHAP Beeswarm)', fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Streamlit App Emulation Test
Verifies that the serialized model, scaler, and features load and predict properly.


In [ ]:
model_path = os.path.join('..', 'Model Training', 'saved_model', 'final_parkinsons_model.pkl')
scaler_path = os.path.join('..', 'Model Training', 'saved_model', 'final_scaler.pkl')

loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)

patient_sample = pd.DataFrame([{
    'UPDRS': 75.0,
    'FunctionalAssessment': 4.5,
    'Tremor': 1,
    'MoCA': 16.0,
    'Bradykinesia': 1,
    'Rigidity': 0
}])[FEATURE_COLS]

scaled_sample = loaded_scaler.transform(patient_sample)
pred = loaded_model.predict(scaled_sample)[0]
prob = loaded_model.predict_proba(scaled_sample)[0]

print('Inference Test Result:')
diag_res = "Parkinson's Disease" if pred == 1 else "Healthy"
print(f"  Prediction: {diag_res}")
print(f"  Confidence: {prob[1]*100:.2f}% risk of Parkinson\'s")
print('Streamlit model compatibility confirmed.')
